# Fine-tune SmolVLA on the S3 pick task (Colab)

Adapted from the official `Finetune_SmolVLA_notebook.ipynb` for Spike S3.

**Version pin (important):** our dataset is LeRobotDataset `codebase_version: v3.0`,
built by lerobot 0.5.2 at commit `58ccc0150867a027e4b3b4ce18dd589113d6ea09`. We pin
Colab's lerobot to the SAME commit so the dataset loads AND the output checkpoint
loads back in our `lerobot-vla` (0.5.2) eval env. Runtime -> Change runtime type -> GPU (A100 preferred).
**Record the exact commit + device you used in `docs/spike_s3_results.md`.**


## 1. condacolab (restarts the runtime once — expected)


In [ ]:
!pip install -q condacolab
import condacolab
condacolab.install()


## 2. Clone + PIN lerobot to our 0.5.2 commit, install


In [ ]:
!git clone https://github.com/huggingface/lerobot.git
!cd lerobot && git checkout 58ccc0150867a027e4b3b4ce18dd589113d6ea09
!conda install -y ffmpeg=7.1.1 -c conda-forge
!cd lerobot && pip install -e ".[smolvla]"


## 3. Log in to Hugging Face (pull the dataset; push the checkpoint later)
Paste a **write** token: https://huggingface.co/settings/tokens


In [ ]:
!hf auth login


## 4. (optional) Weights & Biases for loss curves


In [ ]:
# W&B is OPTIONAL and OFF by default in the train cell below.
# Only run this (and set --wandb.enable=true) if you want W&B logging:
# !wandb login


## 5. Mount Google Drive
Write checkpoints to Drive so they survive a Colab disconnect (and you can resume).


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 6. Train
Set `DATASET` to the repo id you **pushed** (README step 2). Default namespace is
`anikmall`; if you pushed to your own, enter that. Hyperparam notes are in `README.md`.
- A100: `--batch_size=64`. **T4 16 GB: drop to `--batch_size=16`** if it OOMs.
- `--scheduler_decay_steps` must match `--steps` or the LR never decays.
- `--freeze_vision_encoder=true` — keep the encoder frozen; the shipped S3 checkpoint trained with true (corrected 2026-09-04; the run cell below carries the dated note).


In [ ]:
import os
# DATASET must match what you PUSHED in step 2 (record/push/train all agree).
DATASET = (os.environ.get('ARMANI_S3_REPO_ID')
           or input('dataset repo id you pushed [anikmall/armani_pick_red_v1]: ').strip()
           or 'anikmall/armani_pick_red_v1')
HF_USER = DATASET.split('/')[0]  # namespace, reused for the optional checkpoint upload
os.environ['HF_USER'] = HF_USER
OUTPUT  = '/content/drive/MyDrive/s3/smolvla_pick_red'
print('dataset:', DATASET, '| output:', OUTPUT)


In [ ]:
# !! CORRECTED 2026-08-24 — this cell did NOT reproduce the S3 recipe. !!
# The delivered S3 checkpoint (~/models/smolvla_pick_red_v1, rev 85eb875e...) records, in its own
# train_config.json:  freeze_vision_encoder=TRUE, train_expert_only=FALSE, train_state_proj=TRUE.
# This cell previously passed --policy.freeze_vision_encoder=false --policy.train_expert_only=false,
# i.e. it would have trained a DIFFERENT recipe from the baseline. The fallback-3 addendum requires
# arm A to keep the S3 recipe verbatim, so running the old cell would have silently broken parity.
# Old flags kept here, struck through, rather than deleted:
#     --policy.freeze_vision_encoder=false \
#     --policy.train_expert_only=false \
# Memory note: the "A100 preferred / T4 drop to batch 16" advice above was written for the UNFROZEN
# config. With the encoder frozen the footprint is far smaller; batch 64 is what the baseline
# actually ran. Measure before down-scoping.
!cd lerobot && lerobot-train \
  --policy.path=lerobot/smolvla_base \
  --dataset.repo_id={DATASET} \
  --output_dir={OUTPUT} \
  --job_name=smolvla_pick_red \
  --batch_size=64 \
  --steps=20000 \
  --policy.scheduler_decay_steps=20000 \
  --policy.scheduler_warmup_steps=1000 \
  --policy.optimizer_lr=1e-4 \
  --seed=1000 \
  --save_freq=5000 \
  --policy.freeze_vision_encoder=true \
  --policy.train_expert_only=false \
  --policy.train_state_proj=true \
  --policy.device=cuda \
  --wandb.enable=false   # flip to true ONLY after `wandb login` (cell above)


### Resume after a disconnect (only works because output_dir is on Drive)


In [ ]:
# !cd lerobot && lerobot-train \
#   --config_path=/content/drive/MyDrive/s3/smolvla_pick_red/checkpoints/last/pretrained_model/train_config.json \
#   --resume=true


## 7. Get the checkpoint for eval
The last checkpoint is at `<output_dir>/checkpoints/last/pretrained_model/`.
Either download that whole folder from Drive to your Mac, OR push it to the Hub
and `hf download` it on the Mac. That folder is the `--policy-path` for eval.


In [ ]:
CKPT = f'{OUTPUT}/checkpoints/last/pretrained_model'
print('checkpoint folder (config.json + model.safetensors + processor jsons):')
!ls -la {CKPT}
# Optional: push to the Hub for easy download on the Mac:
# !hf upload {HF_USER}/smolvla_pick_red {CKPT} . --repo-type=model --private
